# 02 · Databanken verkennen en aanvullen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Een weegtoestel en een voederstation registreren veel bezoeken. Meestal zal elk toestel, zeker als die van een verschillene fabriekant is, die gegevens in een eigen, aparte databank bewaren.

Je opent in deze oefening een voorbeeld databank op je laptop met **DB Browser for SQLite**. Je onderzoekt de tabellen, en voert stap per stap enkele handelingen uit om met een SQL databank vertrouwt te geraken. Daarna upload je de bewerkte databank naar Codespaces en controleer je of je wijzigingen correct zijn aangekomen.

**Nodig:** je bestaande Codespace en eigen branch uit notebook 01, het opgeslagen bestand `gegevens/inlevering/01_werkplekcontrole.json`, je laptop en DB Browser for SQLite.

Voer de codecellen één voor één uit. Tussen de cellen voer je ook handelingen op je laptop uit. In notebook 01 heb je een rol gekozen. Op basis van die rol kiest dit notebook automatisch jouw databank (`asserva.db` voor gewichten of `schauer.db` voor voederopname). De uitvoer verschijnt wanneer je de codecellen uitvoert.

</div>

## 1. Je werkplek controleren en databank kiezen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 02)

Hervat je bestaande Codespace via [github.com/codespaces](https://github.com/codespaces) en open `notebooks/02_databanken_verkennen_en_aanvullen.ipynb`. Kies rechtsboven opnieuw **Python 3.12** als kernel.

De codecel hieronder leest jouw opgeslagen instellingen uit `gegevens/inlevering/01_werkplekcontrole.json` en controleert of die bij je actieve branch horen. Op basis van de **rol** die je in notebook 01 koos, selecteert de code automatisch het juiste databankbestand:
- rol **`gewichten`** → databank **`asserva.db`**
- rol **`voederopname`** → databank **`schauer.db`**

</div>

In [ ]:
from pathlib import Path
from contextlib import closing
import subprocess
import sys
import re
import json
import sqlite3
import pandas as pd

resultaat = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
if resultaat.returncode:
    raise RuntimeError("Open dit notebook vanuit de repository in Codespaces.")
hoofdmap = Path(resultaat.stdout.strip())
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=hoofdmap, text=True).strip()
if not re.fullmatch(r"student/[a-z0-9][a-z0-9_-]*", branch):
    raise RuntimeError("Selecteer eerst je eigen student/voornaam_achternaam-branch uit notebook 01.")
instellingenpad = hoofdmap / "gegevens/inlevering/01_werkplekcontrole.json"
if not instellingenpad.is_file():
    raise FileNotFoundError("Voer eerst de controlebestandcel van notebook 01 uit en bewaar het resultaat.")
instellingen = json.loads(instellingenpad.read_text(encoding="utf-8"))
if instellingen.get("branch") != branch or instellingen.get("deelnemerscode") != branch.removeprefix("student/"):
    raise ValueError("De instellingen horen bij een andere branch. Voer notebook 01 op je eigen branch opnieuw uit.")
rol = instellingen.get("rol")
bestanden = {"gewichten": "asserva.db", "voederopname": "schauer.db"}
if rol not in bestanden or instellingen.get("bronbestand") != bestanden[rol]:
    raise ValueError("Controleer je rol en bronbestand in notebook 01.")
bestandsnaam = bestanden[rol]
origineel = hoofdmap / "gegevens/origineel" / bestandsnaam
invoermap = hoofdmap / "gegevens/werk/invoer"
invoermap.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(hoofdmap / "programmatuur"))
from databankcontrole import openen, inspecteer, controleer_aanvulling, bewaar_werkbestand
print("Branch:", branch)
print("Rol:", rol)
print("Jouw databank:", bestandsnaam)
print("Upload je bewerkte kopie straks naar: gegevens/werk/invoer/")

Branch: student/rene_vandekerckhove
Rol: voederopname
Jouw databank: schauer.db
Upload je bewerkte kopie straks naar: gegevens/werk/invoer/


## 2. SQL, SQLite en DB Browser

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je laptop (browser en installatieprogramma)

Een **relationele databank** bewaart gegevens in tabellen: een **record** is één rij, een **kolom** beschrijft een eigenschap van die rij (zoals identificatie, datum en gewicht). *Relationeel* betekent dat afzonderlijke tabellen via een gemeenschappelijke code (zoals een diercode `EID` of hokcode `hok`) aan elkaar gekoppeld kunnen worden. Zo hoef je bijvoorbeeld de naam en beschrijving van een hok niet 80.000 keer bij elke meting te herhalen, maar bewaar je die één keer in een aparte hoktabel. Alternatieven zijn bijvoorbeeld een niet-relationele **documentdatabank** zoals MongoDB, waar elk record een los JSON-document met een eigen structuur is, of een los Excel-/CSV-bestand zonder afgedwongen relaties tussen tabellen.

Om met zo'n relationele databank te werken gebruiken we drie onderdelen:

- **SQL (Structured Query Language)** is de standaardtaal om gegevens op te vragen en tabellen aan te passen (`SELECT` om op te halen, `INSERT` om toe te voegen, `UPDATE` om te wijzigen en `DELETE` om te verwijderen). Belangrijke eigenschappen van SQL:
  - **Declaratief (tegenover niet-declaratief):** in SQL beschrijf je *wat* je wilt zien (bijvoorbeeld `SELECT * FROM metingen WHERE PEN = 'A1'`), niet stap voor stap *hoe* de computer moet zoeken. In een *niet-declaratieve* (imperatieve) aanpak, zoals een gewone `for`-lus in Python, moet je zelf programmeren: *‘open het bestand, lees elke rij van boven naar beneden, controleer of `PEN == 'A1'` is en voeg de treffers toe aan een lijst’*. Het gevolg: imperatieve code wordt langer en trager als je miljoenen rijen hebt, tenzij je zelf ingewikkelde zoekstructuren schrijft. Bij SQL kiest de databanksoftware automatisch een snelle route (zoals een index) zonder dat je query hoeft te veranderen.
  - **Vaste structuur en datatypes:** elke kolom heeft een vast type (zoals `TEXT`, `INTEGER` of `REAL`) en regels (zoals een unieke sleutel), zodat tekst en getallen niet per ongeluk door elkaar lopen.
  - **Standaard met eigen dialecten:** de basissyntaxis ligt vast in een internationale SQL-standaard, waardoor gewone `SELECT`- en `INSERT`-opdrachten overal grotendeels hetzelfde zijn. Toch voegen databanksystemen eigen uitbreidingen (*dialecten*) toe, waardoor details verschillen:
    - *Autonummering:* `AUTOINCREMENT` in SQLite tegenover `SERIAL` in PostgreSQL en `AUTO_INCREMENT` in MySQL.
    - *Datums en tekst:* `strftime(...)` in SQLite tegenover `DATE_TRUNC(...)` in PostgreSQL, of hoofdletterongevoelig zoeken (`ILIKE` in PostgreSQL bestaat niet standaard in SQLite).
    - *Datatypes:* SQLite gebruikt slechts enkele basistypes (`TEXT`, `INTEGER`, `REAL`), terwijl PostgreSQL ook aparte types kent zoals `BOOLEAN`, `TIMESTAMP` en `JSONB`.
- **SQLite** is de **databankengine** (*motor*): de achterliggende software die jouw SQL-opdrachten vertaalt naar echte lees- en schrijfacties op schijf, controleert of datatypes en unieke sleutels kloppen en zorgt dat gegevens niet corrupt raken als een programma stopt. Terwijl grotere engines zoals PostgreSQL of MySQL als aparte server op een computer draaien, zit SQLite als compacte bibliotheek rechtstreeks ingebouwd in DB Browser en Python en bewaart het de hele databank in één `.db`-bestand.
- **DB Browser for SQLite** is het gratis grafische programma op je laptop waarmee je zo'n `.db`-bestand opent, tabellen bekijkt en SQL-opdrachten uitvoert.

</div>

### Installeren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

1. Open de [officiële downloadpagina](https://sqlitebrowser.org/dl/).
2. **Windows:** kies het installatieprogramma voor jouw Windows-versie, meestal de 64-bit installer. Open het gedownloade installatiebestand en volg de stappen. Start daarna **DB Browser for SQLite**, niet DB Browser for SQLCipher.
3. **macOS:** kies de macOS-download, open het `.dmg`-bestand en sleep DB Browser naar **Applications**. Open het daarna vanuit Applications.
4. Je ziet een venster met onder andere **Open Database**, **Database Structure**, **Browse Data** en **Execute SQL**.

De schermafbeeldingen in deze oefening zijn gemaakt met DB Browser for SQLite 3.13.1 op Windows, met Engelse menunamen. Op macOS hebben de overeenkomstige menu's en tabbladen dezelfde functie.

</div>

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;white-space:normal;overflow-wrap:anywhere;"><strong>Tip</strong><br>Heb je geen installatierechten op Windows, dan is op de officiële downloadpagina ook een ZIP-versie zonder installatie beschikbaar. Pak die volledig uit en open DB Browser for SQLite vanuit die map. Laat bij een blokkering door instellingsbeheer de toegestane installatie nakijken.</div>

## 3. Je databank downloaden

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Explorer) en op je laptop (bestandenbeheer)

**Waar staat je `.db`-bestand nu en waar moet het naartoe?**
- **Nu:** het startbestand (`asserva.db` of `schauer.db`) staat in Codespaces in de map **`gegevens/origineel/`**.
- **Straks:** je downloadt hiervan een kopie naar jouw eigen laptop, zodat je die daar met DB Browser kunt openen en bewerken.

Beide databanken zijn direct te openen SQLite-bestanden en bevatten de eerste 28 kalenderdagen (**24 augustus tot en met 20 september 2023**).

</div>

| Jouw rol | Download dit bestand | Inhoud van `metingen` |
|---|---|---|
| gewichten | `gegevens/origineel/asserva.db` | 80.028 records met gewichtsgegevens |
| voederopname | `gegevens/origineel/schauer.db` | 146.649 records met voederopnamegegevens |

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

1. Maak op je laptop een map **`practicum_iot`** en daarin een map **`werk`**.
2. Open in Codespaces links **Explorer → gegevens → origineel**. Klik met de rechtermuisknop op jouw `.db`-bestand en kies **Download**.
3. Verplaats het gedownloade bestand naar `practicum_iot/werk/`. Behoud de naam `asserva.db` of `schauer.db`. Staat er `(1)` in de naam, controleer dan welke download je wilt gebruiken en geef die de juiste naam.
4. Start DB Browser, kies **Open Database** en open de databank uit jouw lokale map `werk`.

Het oorspronkelijke bestand blijft in Codespaces onder `gegevens/origineel/` staan. De kopie op je laptop is degene die je nu bewerkt.

</div>

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;white-space:normal;overflow-wrap:anywhere;"><strong>Tip: een .db-bestand of een SQL-export</strong><br>Een <code>.db</code>-bestand bevat de databank zelf: kies <strong>Open Database</strong>. Een <code>.sql</code>-export bevat tekst met SQL-opdrachten en moet eerst via <strong>File → Import → Database from SQL file…</strong> worden ingelezen. Dat werkt alleen als de opdrachten geschikt zijn voor SQLite. In deze oefening ontvang je al een <code>.db</code>-bestand en gebruik je de importstap niet.</div>

## 4. De structuur en meetgegevens onderzoeken

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je laptop (DB Browser)

De structuur bepaalt wat iedere rij voorstelt. Een **primaire sleutel** identificeert een record uniek. In de tabel `metingen` is dat `ROW_ID`. Eén dier kan meerdere bezoeken hebben, daarom kan de dieridentificatie `EID` hier geen unieke sleutel voor een bezoek zijn.

1. Open **Database Structure** en klap **Tables** open.
2. Je ziet **`metingen`** en **`metadata`**. Klap `metingen` open om de kolommen te bekijken.
3. Zoek `ROW_ID`, `EID`, `DATE` en jouw meetveld: **`VBW`** voor gewichten of **`VFI`** voor voederopname.
4. Bekijk het gegevenstype. **TEXT** bewaart tekst, **REAL** een getal met mogelijke decimalen en **INTEGER** een geheel getal. Bij `ROW_ID` staan ook **PRIMARY KEY** en **NOT NULL**: de identificatie moet uniek en ingevuld zijn.

De oorspronkelijke kolomnamen zijn behouden, zodat je de koppeling met de aangeleverde bronbestanden kunt volgen. De bestandsnamen verwijzen naar de toestellen. Dit is een onderwijsstructuur, geen officiële fabrikantendatabase.

</div>

![De twee tabellen van de startdatabank](../afbeeldingen/schermafbeeldingen/02_databankstructuur.png)

*Figuur 1. Database Structure in DB Browser for SQLite 3.13.1 op Windows. Met het pijltje naast een tabel klap je de kolommen open.*

| Veld | Betekenis | Bewaarwijze / eenheid |
|---|---|---|
| `ROW_ID` | Identificatie van het bronrecord | TEXT, 27 of 28 cijfers |
| `EID` | Dieridentificatie | TEXT, hetzelfde dier kan meerdere bezoeken hebben |
| `PEN` | Hokcode in de bron | TEXT |
| `DATE` | Kalenderdatum waarop de bronselectie is gebaseerd | TEXT, `JJJJ-MM-DD` |
| `VBW` | Lichaamsgewicht bij een gewichtsrecord | REAL, kg |
| `VFI` | Voederopname per bezoek | REAL, g |
| `DATAORIGIN` | Herkomstcode, bijvoorbeeld `imputed` | TEXT, broncode behouden |
| `FILTER`, `ERROR`, `FLAG_…` | Kwaliteitsinformatie uit de verwerking | TEXT, broncodes behouden |

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

De volledige veldtoelichting staat in [het gegevenswoordenboek](../gegevens/gegevenswoordenboek.md).

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Nu je de kolomnamen en datatypes kent, bekijk je de echte meetgegevens in het tabblad **Browse Data**:

5. Klik bovenaan op het tabblad **Browse Data** en kies in het keuzemenu **Table:** de tabel **`metingen`**.
6. Scroll verticaal door de rijen (of gebruik onderaan de paginaknoppen en **Go to:**) en scroll horizontaal naar rechts om kolommen zoals `VBW` of `VFI`, `DATAORIGIN`, `ERROR` en `FILTER` te bekijken.
7. Probeer ook de zoekfuncties in **Browse Data**:
   - **Sorteren:** klik op een kolomkop (bijvoorbeeld `DATE` of `VBW`) om de rijen op die kolom te sorteren.
   - **Snel filteren:** typ in het lege vakje direct onder een kolomkop (de filterrij) een zoekwaarde, zoals een datum (`2023-08-25`) of een diercode (`EID`), zodat alleen die rijen getoond worden.
   - **Volledige waarde lezen:** klik op een cel met een lange code (zoals `ROW_ID`) om de volledige tekst rechts in **Edit Database Cell** onafgekapt te bekijken.
8. Kies tot slot **`metadata`** in het keuzemenu **Table:**. Deze kleine tabel beschrijft onder andere de rol, de periode, de eenheid en de herkomst van de databank.

</div>

![Gegevens bekijken en filteren in Browse Data](../afbeeldingen/schermafbeeldingen/02_gegevens_bekijken.png)

*Figuur 2. Browse Data in DB Browser for SQLite 3.13.1 op Windows. Bovenaan selecteer je de tabel, onder elke kolomkop kun je filteren, en rechts lees je in Edit Database Cell de volledige waarde van een gekozen cel.*

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;white-space:normal;overflow-wrap:anywhere;"><strong>Uitleg</strong><br>Het recordnummer is een herkenningscode voor elke unieke rij informatie in de tabel. Hier is het van belang dat het gegevenstype TEXT is en bijvoorbeeld niet REAL. Met TEXT blijven alle 27 of 28 cijfers behouden, en worden ze per vergissing niet afgerond. Excel kan bijvoorlbeeld lange codes afronden wanneer het ze als getal behandelt.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 1.** Welke twee tabellen bevat jouw startbestand? Hoeveel kolommen heeft `metingen`, wat is de primaire sleutel en waarom is `EID` niet geschikt als primaire sleutel?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 1):**

metadata en metingen. 24 kolommen, primaire sleutel is ROW_ID, EID is niet geschikt omdat één dier meerdere bezoeken en dus meerdere meetrecords kan hebben.

</div>

## 5. Met SQL gericht gegevens opvragen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je laptop (DB Browser → Execute SQL)

Met een **query** (zoekopdracht) vraag je precies de gegevens op die je nodig hebt, zonder de databank te veranderen. Een SQL-opvraagopdracht volgt altijd dezelfde vaste opbouw van boven naar beneden:

</div>

```sql
SELECT   kolom_1, kolom_2 AS eigen_naam
FROM     tabelnaam
WHERE    voorwaarde
GROUP BY kolom
ORDER BY kolom ASC
LIMIT    aantal;
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Elk onderdeel heeft een vaste taak in deze volgorde:

- **`SELECT`** bepaalt wat je wilt zien: specifieke kolommen (zoals `EID, VBW`), alle kolommen (`*`), of een berekening zoals `COUNT(*)` (aantal rijen), `MIN` (kleinste waarde), `MAX` (grootste waarde) of `AVG` (gemiddelde).
- **`AS`** (optioneel binnen `SELECT`) geeft een kolom of berekening een eigen, leesbare naam (*alias*) in het resultaat, bijvoorbeeld `COUNT(*) AS aantal_records`.
- **`FROM`** geeft aan uit welke tabel de gegevens worden gehaald.
- **`WHERE`** (optioneel) filtert de rijen op basis van een voorwaarde, bijvoorbeeld `WHERE DATE = '2023-08-24'`.
- **`GROUP BY`** (optioneel) groepeert rijen met dezelfde waarde (bijvoorbeeld per dier `GROUP BY EID`), zodat berekeningen zoals `COUNT` of `AVG` per groep worden uitgevoerd.
- **`ORDER BY`** (optioneel) sorteert het resultaat oplopend (`ASC`, standaard) of aflopend (`DESC`).
- **`LIMIT`** (optioneel) beperkt het aantal rijen dat op het scherm verschijnt.

*(SQL-trefwoorden zoals `SELECT` en `FROM` worden voor de leesbaarheid meestal in hoofdletters geschreven. De puntkomma `;` sluit de SQL-opdracht af.)*

### Voorbeeld 1: aantal metingen en periode controleren

Met de eerste opdracht controleer je of je het volledige bestand hebt geopend:

1. Open het tabblad **Execute SQL**.
2. Kopieer de onderstaande SQL-opdracht naar het grote tekstvak.
3. Klik op **Execute SQL** (het afspeelknopje, of **F5**). Selecteer geen losse regel, want zonder tekstselectie voert DB Browser de volledige opdracht uit.
4. Lees de resultaatrij onder het tekstvak.

**In gewone taal betekent deze query:** *‘Tel alle rijen in de tabel `metingen` en noem die uitkomst `aantal_records`, zoek de vroegste waarde in `DATE` en noem die `eerste_datum`, en zoek de laatste waarde in `DATE` en noem die `laatste_datum`.’*

</div>

```sql
SELECT COUNT(*) AS aantal_records,
       MIN(DATE) AS eerste_datum,
       MAX(DATE) AS laatste_datum
FROM metingen;
```

![Query met aantal en datumgrenzen](../afbeeldingen/schermafbeeldingen/02_query.png)

*Figuur 3. Het SQL-tekstvak, de uitvoerknop en het queryresultaat voor de gewichtsdatabank, in DB Browser 3.13.1 op Windows.*

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Je verwacht 80.028 records (bij `asserva.db`) of 146.649 records (bij `schauer.db`), met als uiterste datums `2023-08-24` en `2023-09-20`.

### Voorbeeld 2: de eerste vijf rijen netjes gesorteerd opvragen

Vervang de tekst daarna door deze tweede query.

**In gewone taal betekent deze query:** *‘Haal uit de tabel `metingen` alleen de kolommen `ROW_ID`, `EID`, `DATE` en `FILTER`, sorteer de rijen oplopend op datum (`DATE`) en bij gelijke datum op `ROW_ID`, en toon alleen de eerste vijf rijen.’*

</div>

```sql
SELECT ROW_ID, EID, DATE, FILTER
FROM metingen
ORDER BY DATE, ROW_ID
LIMIT 5;
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 2.** Welke aantallen en datums geeft jouw eerste query? Betekent het resultaat van de tweede query dat de databank nog maar vijf meetrecords bevat?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 2):**

De eerste query geeft in totaal 146.649 meetrecords weer. De metingen lopen van 24 augustus 2023 tot en met 20 september 2023. de tweede query toont slechts de eerste vijf meetrecords, gesorteerd op datum en ROW_ID. Dit komt door LIMIT 5, dat het aantal weergegeven resultaten beperkt.

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 3.** Welke eenheid hoort bij jouw meetveld? Wat betekent het wanneer een gewicht als `imputed` is gemarkeerd, en waarom bewaar je zo’n markering?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 3):**

mijn meetveld VFI wordt uitgedrukt in gram. Imputed betekent dat eeen ontbrekende meetwaarde geschat of berekend werd. de markering blijft behouden om gemeten en geschatte waarden te onderscheiden

</div>

## 6. Hokken opzoeken en een metadatatabel maken

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je laptop (DB Browser → Execute SQL, daarna Database Structure)

Elke rij in `metingen` is één bezoek van een big, waarbij de kolom `PEN` aangeeft in welk hok dat dier zat. Omdat elk hok dus duizenden keren voorkomt, vertelt het totaal aantal meetrecords (`COUNT(*)`) niet hoeveel hokken er zijn.

Om het echte aantal hokken en de bijbehorende hoknummers te vinden, gebruik je `DISTINCT` (unieke waarden). Voer eerst deze twee queries één voor één uit in **Execute SQL** en noteer het aantal en de hoknummers hieronder:

</div>

```sql
SELECT COUNT(DISTINCT PEN) AS aantal_hokken
FROM metingen;
```

```sql
SELECT DISTINCT PEN AS hok
FROM metingen
ORDER BY CAST(PEN AS INTEGER);
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Met `DISTINCT` verschijnt elke hokcode maar één keer. Omdat `PEN` in `metingen` als tekst is opgeslagen, zou een gewone sortering de volgorde `1, 10, 2, 3, ...` geven (alfabetisch). Door `CAST(PEN AS INTEGER)` sorteert SQL de hokcodes netjes als getallen (`1, 2, ..., 10`), zonder het opgeslagen datatype te veranderen.

Gegevens over het hok zelf (zoals het gebouw en de oppervlakte) noemen we **metadata**: achtergrondinformatie bij de metingen. Die zet je niet bij elk van de duizenden bezoeken opnieuw, maar in een aparte tabel **`hokken`** met één rij per hok. Via de gedeelde hokcode (`hokken.hok` = `metingen.PEN`) kunnen we beide tabellen later weer aan elkaar koppelen.

Maak nu de tabel **`hokken`** aan:
1. Ga naar **Database Structure → Create Table**.
2. Vul bij **Table** de naam **`hokken`** in.
3. Voeg met **Add** de vier velden uit de tabel hieronder toe, in deze volgorde. Je vult in DB Browser alleen **Name**, **Type**, **NN** en **PK** in (de kolom *Betekenis* hieronder is alleen ter uitleg).
4. Vink bij `hok` zowel **PK** (Primary Key) als **NN** (Not Null) aan. Laat **AI** (Auto Increment) uit, want je neemt straks de bestaande hoknummers over.
5. Controleer de namen en types en klik op **OK**. Onderaan het venster zie je meteen de SQL-opdracht die DB Browser hiervoor gebruikt.

</div>

| Name | Type | NN | PK | Betekenis (alleen uitleg, niet invullen) |
|---|---|---|---|---|
| `hok` | TEXT | aan | aan | Hoknummer, dezelfde code als PEN |
| `gebouw` | TEXT | uit | uit | Naam van de stal |
| `afdeling` | INTEGER | uit | uit | Nummer van de afdeling |
| `oppervlakte_m2` | REAL | uit | uit | Hokoppervlakte in vierkante meter |

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

De hokcode blijft tekst (`TEXT`), net zoals `PEN` in `metingen`. Het type `REAL` zorgt ervoor dat je straks ook oppervlakten met decimalen kunt invullen. De eenheid staat al in de kolomnaam, dus vul bij `oppervlakte_m2` alleen een getal in (zonder `m²`).

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Sectie 6):**

Mijn databank bevat 10 verschillende hokken, genummerd van 1 tot en met 10.

</div>

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;white-space:normal;overflow-wrap:anywhere;"><strong>Tip: verderwerken</strong><br>Bestaat <code>hokken</code> al omdat je de oefening hervat? Controleer dan de kolommen met <strong>Modify Table</strong> en werk verder met die tabel. Maak geen tweede tabel met bijna dezelfde naam.</div>

## 7. Hokmetadata handmatig toevoegen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je laptop (DB Browser → Browse Data)

**Opdracht:** voeg voor elk hoknummer uit je query één rij toe aan `hokken`. Alle tien hokken liggen in **stal M5**, **afdeling 1**, en hebben een **oppervlakte van 8 m²**. Vul bij `gebouw` dus `M5` in, bij `afdeling` het getal `1` en bij `oppervlakte_m2` het getal `8`. Neem de hoknummers over uit jouw queryresultaat.

1. Ga naar **Browse Data** en kies `hokken` bij **Table**.
2. Klik op **New Record** (of **Insert Record**).
3. Krijg je een melding `NOT NULL constraint failed: hokken.hok`? Dat is normaal in DB Browser 3.13.1 omdat `hok` niet leeg (`NULL`) mag blijven. Klik op **OK**.
4. In het venster **Add New Record** (of rechtstreeks in de tabel) vul je de vier waarden in: dubbelklik in de kolom **Value** op `NULL` naast `hok`, `gebouw`, `afdeling` en `oppervlakte_m2`, typ de waarde in (voor hok 1 dus `1`, `M5`, `1`, `8`) en klik op **Save**.
5. Voeg op dezelfde manier een rij toe voor alle volgende hoknummers (`2` tot en met `10`). Wil je later een bestaande waarde aanpassen, klik dan op die cel in de tabel, wijzig hem rechts bij **Edit Database Cell** en klik op **Apply**.
6. Vergelijk alle tien hoknummers en waarden met de opdracht en klik bovenaan op **Write Changes** om ze in het bestand op te slaan.

Werk je later verder aan dit bestand? Verbeter dan bestaande rijen en voeg ze niet dubbel toe. De primaire sleutel houdt dubbele hokcodes tegen, maar controleert niet vanzelf of alle hokken uit `metingen` aanwezig zijn. Of jouw lijst compleet is, controleer je dus zelf hieronder (en later automatisch met Python).

</div>

### De inhoud controleren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je laptop (DB Browser → Execute SQL)

Vervang de vorige query door:

</div>

```sql
SELECT hok, gebouw, afdeling, oppervlakte_m2
FROM hokken
ORDER BY CAST(hok AS INTEGER);
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Voer de query uit en vergelijk het aantal rijen en de hoknummers met je eerdere zoekopdracht. Controleer of bij elk hok gebouw `M5`, afdeling `1` en oppervlakte `8` staat.

**Zelf proberen:**
1. Pas de query in DB Browser aan met `WHERE` tussen `FROM` en `ORDER BY`, zodat alleen hokken met een oppervlakte **groter dan 8 m²** verschijnen (`>`). Voorspel eerst hoeveel rijen je krijgt.
2. Verander die voorwaarde daarna in **groter dan of gelijk aan 8 m²** (`>=`) en vergelijk beide resultaten.
3. Bewaar die tweede query hieronder in de codecel (terwijl je de SQL zelf in DB Browser uitvoert).

</div>

In [ ]:
# Bewaar hier jouw SQL-query als tekst. Voer de query uit in DB Browser.
SELECT hok, gebouw, afdeling, oppervlakte_m2
FROM hokken
WHERE oppervlakte_m2 >= 8
ORDER BY CAST(hok AS INTEGER);

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 4.** Hoeveel hokken verschijnen bij `oppervlakte_m2 > 8` en bij `oppervlakte_m2 >= 8`? Zijn er records verwijderd wanneer de eerste query geen rijen toont?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 4):**

0.  10. er worden geen records verwijderd wanneer de eerste query geen rijen toont. WHERE filtert enkel de resultaten die worden weergegeven; de gegevens blijven in de databank staan.

</div>

### Simultaan gegevens opvragen en controleren met `JOIN`

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je laptop (DB Browser → Execute SQL)

Je hebt nu twee aparte tabellen in dezelfde databank: `metingen` (met de metingen en `PEN`) en `hokken` (met de metadata en `hok`). Met een **`JOIN`** leg je nog **geen blijvende koppeling** vast in de databank zelf, maar voer je een **gekoppelde zoekopdracht** uit. SQL leest daarbij simultaan uit beide tabellen en combineert de rijen alleen voor dit queryresultaat. Zo kun je meteen controleren of alle gegevens uit beide tabellen netjes op elkaar aansluiten.

- `FROM metingen AS m` en `JOIN hokken AS h` geven beide tabellen een korte bijnaam (`m` en `h`), zodat je niet telkens de volledige tabelnaam hoeft te typen.
- `ON m.PEN = h.hok` vertelt SQL welke rijen bij elkaar horen: bij elke meetrij zoekt SQL in `hokken` de rij waarvan `hok` gelijk is aan `PEN`.

**In gewone taal betekent deze query:** *‘Haal de eerste tien metingen uit `metingen` (`m`), zoek bij elke meting de bijbehorende rij uit `hokken` (`h`) waar `m.PEN = h.hok`, en toon per meting simultaan het gebouw, de afdeling en de oppervlakte mee.’*

Je hoeft hiervoor geen extra gegevens in een tabel in te voeren (als de 10 rijen in `hokken` klaarstaan, werkt het meteen):
1. Open in DB Browser het tabblad **Execute SQL**.
2. Vervang de tekst in het grote SQL-vak door de onderstaande query.
3. Klik op **Execute SQL** (het afspeelknopje, of **F5**) en bekijk het gecombineerde resultaat.

</div>

```sql
SELECT m.ROW_ID, m.EID, m.DATE, m.PEN,
       h.gebouw, h.afdeling, h.oppervlakte_m2
FROM metingen AS m
JOIN hokken AS h ON m.PEN = h.hok
ORDER BY m.DATE, m.ROW_ID
LIMIT 10;
```

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Controleer daarna op dezelfde plek (in het tabblad **Execute SQL**) met een **`LEFT JOIN`** of er nog metingen zijn waarvan het hoknummer ontbreekt in `hokken`. Bij een `LEFT JOIN` blijven alle rijen uit `metingen` bewaard, ook als er geen overeenkomst in `hokken` wordt gevonden (dan blijft `h.hok` leeg, oftewel `NULL`). Vervang de query in het SQL-vak door onderstaande opdracht en klik opnieuw op **Execute SQL** (**F5**). Als je alle tien hokken goed hebt ingevuld, geeft dit **`0`** terug.

</div>

```sql
SELECT COUNT(*) AS metingen_zonder_hok
FROM metingen AS m
LEFT JOIN hokken AS h ON m.PEN = h.hok
WHERE h.hok IS NULL;
```

### Een vaste koppeling (`Foreign Key`) instellen, bekijken en weer verwijderen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je laptop (DB Browser)

Met een `JOIN` voer je alleen tijdens het opvragen een gekoppelde zoekopdracht uit, zonder dat er een vaste relatie in de databank zelf is opgeslagen. Wil je wél een blijvende koppeling in de databankstructuur vastleggen, zodat de databank automatisch afdwingt dat elk hoknummer moet bestaan in `hokken`, dan stel je een **Foreign Key** (*verwijzende sleutel*) in.

Omdat we de grote tabel `metingen` ongewijzigd willen houden voor de eindcontrole, oefen je dit kort op een kleine tijdelijke tabel **`meting_test`** die verwijst naar jouw nieuwe tabel `hokken`. Daarna verwijder je de Foreign Key en de testtabel weer.

**1. De Foreign Key installeren:**
1. Ga naar **Database Structure → Create Table** en vul als tabelnaam **`meting_test`** in.
2. Voeg met **Add** twee kolommen toe:
   - `id` met type `INTEGER`, en vink **NN** en **PK** aan.
   - `hok` met type `TEXT`.
3. Scroll bij de rij `hok` helemaal naar rechts naar de kolom **Foreign Key**, dubbelklik in dat vakje, kies tabel **`hokken`** en kolom **`hok`** (zodat er `"hokken"("hok")` staat) en klik op **OK**. Het extra veld **Foreign key clauses** laat je leeg. *(Met zo'n clause geef je optioneel aan wat er moet gebeuren als een hok uit `hokken` later wordt verwijderd of gewijzigd terwijl er nog metingen naar verwijzen, zoals `ON DELETE CASCADE` om gekoppelde rijen automatisch mee te verwijderen of `ON DELETE SET NULL` om het veld leeg te maken. Laat je dit leeg, dan houdt SQLite verwijderen of wijzigen standaard tegen zolang er nog gekoppelde rijen bestaan.)*

**2. De koppeling visueel controleren (en optioneel testen):**
1. Kijk in **Database Structure** in de kolom **Schema** achter `meting_test`. Je ziet nu letterlijk `FOREIGN KEY("hok") REFERENCES "hokken"("hok")` staan.
2. Klik rechtsonder op het tabblad **DB Schema** (naast `SQL Log`, `Plot` en `Remote`) en klap `meting_test` open om de kolommen en verwijzingen te zien. *(DB Browser heeft geen diagram met getekende lijntjes tussen tabellen, maar toont vaste koppelingen als tekst in **Schema**, **DB Schema** en **Modify Table**.)*
3. *(Optioneel testen)* Wil je de beveiliging in actie zien? Open het tabblad **Edit Pragmas**, vink **Foreign Keys** aan en klik op **Save** (of **Apply**). Voeg daarna in **Browse Data** bij `meting_test` een rij met `id = 1` en `hok = 1` toe (dat lukt), en probeer vervolgens `id = 2` met `hok = 99`. SQLite weigert die tweede rij meteen met `FOREIGN KEY constraint failed`, omdat hok `99` niet bestaat in `hokken`.

**3. De Foreign Key en de testtabel weer verwijderen:**
1. Klik in **Database Structure** op `meting_test` en kies **Modify Table**.
2. Wis de verwijzing in de kolom **Foreign Key** bij `hok` en klik op **OK**. Controleer in de kolom **Schema** en in **DB Schema** rechtsonder dat de `FOREIGN KEY`-regel verdwenen is.
3. Selecteer daarna `meting_test` in **Database Structure**, klik op **Delete Table** (bevestig met **Yes**) en klik bovenaan op **Write Changes**, zodat alleen `metingen`, `metadata` en `hokken` overblijven.

</div>

## 8. Het alternatief voor veel rijen: CSV-import

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je laptop (DB Browser)

Tien rijen kun je nog makkelijk met de hand invoeren en nakijken, maar bij duizenden rijen is dat geen optie. Een **CSV-bestand** bewaart een tabel als platte tekst met een scheidingsteken tussen de kolommen. DB Browser kan CSV rechtstreeks importeren, terwijl je een Excel-werkblad daarvoor eerst als CSV moet opslaan.

Vergelijk beide werkwijzen nu met je ingevulde hokmetadata:

1. Kies **File → Export → Table(s) as CSV file…**, selecteer `hokken` en bewaar het bestand als `hokken.csv` op je laptop. Vink aan dat de kolomnamen op de eerste regel moeten staan.
2. Kies **File → Import → Table from CSV file…** en open dat bestand.
3. Geef de vergelijkingstabel de naam **`hokken_csv`**, zodat je handmatig gemaakte tabel `hokken` behouden blijft.
4. Vink **Column names in first line** aan en kies **UTF-8** en het scheidingsteken uit je export. Controleer in het voorbeeld de vier kolommen `hok`, `gebouw`, `afdeling` en `oppervlakte_m2`.
5. Bevestig de import, vergelijk in **Browse Data** de tien rijen met `hokken` en klik daarna op **Write Changes**.

Bestaat `hokken_csv` al omdat je verderwerkt, bekijk die tabel dan gewoon opnieuw. Kijk tot slot ook even in **Database Structure**: een CSV-bestand bewaart geen primaire sleutels of datatypes, waardoor de geïmporteerde tabel andere veldeigenschappen heeft dan jouw zelfgemaakte tabel `hokken`.

</div>

<div style="background-color:#e3f2fd;color:#0d47a1;border-left:5px solid #1976d2;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;white-space:normal;overflow-wrap:anywhere;"><strong>Tip: zelf exporteren vanuit Excel</strong><br>Je kunt de tien hokrecords en hun kolomkoppen ook naar een apart werkblad kopiëren en dat als CSV UTF-8 opslaan. Kies in DB Browser het scheidingsteken dat je in het voorbeeld ziet: Nederlandstalige Excel-instellingen gebruiken soms een puntkomma. Een CSV bevat één tabel en bewaart geen werkbladopmaak.</div>

## 9. Opslaan, sluiten en uploaden

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** op je laptop (DB Browser), daarna in Codespaces (Explorer)

**Waar staat je `.db`-bestand nu en waar gaat het naartoe?**
- **Nu:** jouw bewerkte databank staat op jouw **laptop** in `practicum_iot/werk/`.
- **In deze stap:** eerst schrijf je de wijzigingen weg in DB Browser en sluit je het bestand. Daarna upload je die bewerkte kopie naar de ontvangstmap **`gegevens/werk/invoer/`** in **Codespaces** (dus *niet* naar `gegevens/origineel/`).

DB Browser bewaart bewerkingen eerst in het geheugen voordat ze in het `.db`-bestand terechtkomen. Door eerst op **Write Changes** te klikken en de databank te sluiten, weet je zeker dat je de volledige, bijgewerkte versie uploadt.

1. Klik op **Write Changes** bovenaan DB Browser. **Save Project** is iets anders: dat bewaart venster- en projectinstellingen, niet de vervanging voor Write Changes.
2. Kies **Close Database**.
3. Open dezelfde lokale databank nogmaals met **Open Database**. Controleer via Browse Data dat de tien rijen in `hokken` er nog staan. Sluit de databank daarna opnieuw.
4. Ga terug naar Codespaces. Open in Explorer **gegevens → werk → invoer**.
5. Klik met de rechtermuisknop op **invoer**, kies **Upload…** en selecteer de gesloten databank uit `practicum_iot/werk/` op je laptop. Je kunt het bestand ook naar die map slepen.
6. Controleer dat de bestandsnaam exact overeenkomt met de naam die stap 1 van dit notebook afdrukte.

De map `invoer` is een ontvangstmap. In de volgende stap vergelijken we de upload met het origineel voordat die jouw werkdatabase wordt. Upload dus naar **`gegevens/werk/invoer/`**, niet naar `gegevens/origineel/`.

</div>

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;white-space:normal;overflow-wrap:anywhere;"><strong>Controle vóór je verdergaat</strong><br>De lokale database is opgeslagen en gesloten. In Codespaces staat jouw bewerkte <code>asserva.db</code> of <code>schauer.db</code> onder <code>gegevens/werk/invoer/</code>. Voer pas daarna de volgende cel uit.</div>

## 10. De overgebrachte databank controleren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 02)

Jouw geüploade `.db`-bestand staat nu in Codespaces in de ontvangstmap **`gegevens/werk/invoer/`**. De cel hieronder opent dat bestand (alleen om te lezen) en controleert of alle oorspronkelijke meetrecords en metadata intact zijn gebleven en of jouw tien nieuwe hokrecords in `hokken` kloppen. Daarna wordt het bestand in de volgende stap gekopieerd naar **`gegevens/werk/`** als jouw vaste werkdatabase voor de volgende notebooks.

</div>

In [ ]:
kandidaat = invoermap / bestandsnaam
controle = controleer_aanvulling(origineel, kandidaat, rol)
print("De meetgegevens en metadata zijn ongewijzigd.")
print("Meetrecords:", controle["aantal_metingen"])
print("Periode:", controle["eerste_datum"], "tot en met", controle["laatste_datum"])
print("Hokrecords:", controle["aantal_hokken"])
with closing(openen(kandidaat)) as db:
    display(pd.read_sql_query("SELECT * FROM hokken ORDER BY CAST(hok AS INTEGER)", db))

<div style="background-color:#e8f5e9;color:#1b5e20;border-left:5px solid #2e7d32;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;white-space:normal;overflow-wrap:anywhere;"><strong>Interpretatie</strong><br>Je ziet tien hokrecords en het oorspronkelijke aantal meetrecords voor jouw databank. Je hebt de databank uitgebreid met contextgegevens en de meettabel is intact gebleven. Verschijnt een foutmelding, herstel dan de aangegeven stap en upload de opnieuw opgeslagen kopie.</div>

### De gecontroleerde kopie als werkdatabase bewaren

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

Voor de volgende oefeningen krijgt de gecontroleerde database een vaste plaats: **`gegevens/werk/asserva.db`** of **`gegevens/werk/schauer.db`**. De cel gebruikt daarvoor de back-upfunctie van SQLite.

Als daar al een identieke werkdatabase staat, blijft die behouden. Staat er een andere versie, dan stopt de cel: je bekijkt eerst welke versie de nieuwste is. Zo kan een oude laptopkopie geen nieuwere werkdatabase vervangen.

</div>

In [ ]:
werkdatabase = hoofdmap / "gegevens/werk" / bestandsnaam
verslag = bewaar_werkbestand(origineel, kandidaat, werkdatabase, rol)
print(verslag["overname"])
print("Werkdatabase:", werkdatabase.relative_to(hoofdmap))
with closing(openen(werkdatabase)) as db:
    display(pd.read_sql_query("PRAGMA table_info(hokken)", db))

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Vraag 5.** Waarom upload je eerst naar `gegevens/werk/invoer/` en controleer je daarna de inhoud? Waarom blijft je werkdatabase buiten Git?

</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Jouw antwoord (Vraag 5):**

*Voeg hier je antwoord in. Tip: verbind dit met het verschil tussen je laptopkopie, werkdatabase en inleverbestanden.*

</div>

## 11. Je resultaat bewaren en afsluiten

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

**Uitvoeren:** in Codespaces (Notebook 02 en Source Control)

Je bewaart een klein controleverslag met je rol, aantallen en bestandslocatie. Zo kun je later aantonen welke stap gelukt is, zonder de hele werkdatabase telkens opnieuw naar GitHub te sturen.

</div>

In [ ]:
inlevering = hoofdmap / "gegevens/inlevering/02_databankcontrole.json"
inlevering.parent.mkdir(parents=True, exist_ok=True)
resultaat = {
    "deelnemerscode": instellingen["deelnemerscode"],
    "branch": branch,
    "rol": rol,
    "werkdatabase": werkdatabase.relative_to(hoofdmap).as_posix(),
    "aantal_metingen": verslag["aantal_metingen"],
    "aantal_hokken": verslag["aantal_hokken"],
    "basis_vanaf": verslag["eerste_datum"],
    "basis_tot_en_met": verslag["laatste_datum"],
    "metingen_sha256": verslag["metingen_sha256"],
}
inlevering.write_text(json.dumps(resultaat, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
assert json.loads(inlevering.read_text(encoding="utf-8")) == resultaat
print("Bewaard:", inlevering.relative_to(hoofdmap))
print("Je kunt nu opslaan, committen en pushen.")

<div style="background-color:#fff8e1;color:#6d4c00;border-left:5px solid #f9a825;padding:14px;margin:12px 0;border-radius:4px;text-align:justify;box-sizing:border-box;max-width:100%;white-space:normal;overflow-wrap:anywhere;"><strong>Stop je Codespace na het practicum</strong><br>Je beschikbare gratis Codespaces-uren zijn beperkt. Bewaar je werk met de stappen hieronder en kies daarna <strong>Stop Current Codespace</strong>, of stop de werkplek via <a href="https://github.com/codespaces">github.com/codespaces</a>. Alleen de browsertab sluiten is niet hetzelfde als stoppen.</div>

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

1. Vul je antwoorden in en kies **File → Save All**.
2. Open **Source Control**. Selecteer dit notebook en `gegevens/inlevering/02_databankcontrole.json` met het **+**-teken naast elk bestand.
3. De werkdatabase onder `gegevens/werk/` hoort niet in de commitlijst te staan. De oorspronkelijke databank onder `gegevens/origineel/` hoort niet als gewijzigd te verschijnen.
4. Commit met bijvoorbeeld **Databank onderzocht en oefentabel toegevoegd** en kies **Push**.
5. Controleer op GitHub op je eigen branch of het notebook en het controleverslag zichtbaar zijn.
6. Download de gesloten werkdatabase uit `gegevens/werk/` naar een aparte map `practicum_iot/bewaard_na_02/` op je laptop. Zo heb je naast de Codespace ook een bewaarde kopie. Houd de mapnaam bij de kopie zodat je de versie herkent.
7. Stop je Codespace. Hervat bij de volgende sessie dezelfde werkplek.

</div>

### Eindcontrole

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

- [ ] Ik kan SQL, SQLite en DB Browser van elkaar onderscheiden.
- [ ] Ik heb tabellen, kolommen, primaire sleutel, eenheid en aantallen opgezocht.
- [ ] Mijn tabel hokken bevat alle tien hoknummers, met gebouw M5, afdeling 1 en oppervlakte 8 m².
- [ ] Ik kan unieke hokken tellen en het effect van > en >= in een WHERE-voorwaarde verklaren.
- [ ] Ik kan met een gekoppelde zoekopdracht (`JOIN`) simultaan gegevens uit `metingen` en `hokken` opvragen en controleren of alle hokken overeenkomen.
- [ ] Ik heb een Foreign Key ingesteld, visueel gecontroleerd in DB Browser en daarna weer verwijderd.
- [ ] Ik heb de CSV-import vergeleken met handmatige invoer.
- [ ] De controle in Codespaces slaagt en mijn vaste werkdatabase is bewaard.
- [ ] Mijn notebook en controleverslag staan op mijn eigen GitHub-branch.
- [ ] Mijn Codespace is gestopt.

</div>

## Hulp bij veelvoorkomende problemen

| Probleem | Wat controleer je? |
|---|---|
| Het notebook of de nieuwe bestanden ontbreken | Je branch heeft de nieuwe lesbestanden nog niet. Bewaar eerst je werk en laat de nieuwe bestanden aan je branch toevoegen. Maak niet zomaar een nieuwe lege werkplek. |
| Het instellingenbestand (`01_werkplekcontrole.json`) ontbrekt | Doorloop de controlebestandcel van notebook 01 op je eigen branch. |
| DB Browser vraagt een wachtwoord | Open de gewone DB Browser for SQLite en het aangeleverde `.db`-bestand. De lesdatabase is niet versleuteld. |
| `no such table` bij een query | Controleer de geopende databank en de naam in de Table-keuzelijst. |
| De upload wordt niet gevonden | Controleer `gegevens/werk/invoer/`, de bestandsnaam en eventuele `(1)` in de downloadnaam. |
| De tien hokrecords ontbreken na upload | Open de laptopkopie, kies Write Changes, sluit de database en upload opnieuw. |
| `NOT NULL constraint failed: hokken.hok` bij New Record | Dit gebeurt in DB Browser 3.13.1 omdat `hok` verplicht is (`NN`). Klik op **OK** en vul in het venster **Add New Record** alle vier waarden in de kolom **Value** in voordat je op **Save** klikt. |
| Een primaire-sleutelfout bij invoer | De hokcode bestaat al. Verbeter de bestaande rij in plaats van een dubbele toe te voegen. |
| De controle meldt gewijzigde meetgegevens | Vergelijk met je originele download. Maak zo nodig een nieuwe lokale werkkopie en voeg alleen de oefentabel toe. Bewaar de afwijkende kopie apart als je die wilt onderzoeken. |
| De werkdatabase bestaat al en verschilt | Bewaar de bestaande werkdatabase apart en bepaal welke versie je nodig hebt. Verwijder geen bestand om de melding zomaar te laten verdwijnen. |
| Je werkt in een nieuwe Codespace | Je gepushte antwoorden zijn beschikbaar. Upload je apart bewaarde database naar de ontvangstmap en voer de controles opnieuw uit. |

### Technische bronnen

<div style="text-align:justify;box-sizing:border-box;max-width:100%;overflow-wrap:anywhere;">

- [DB Browser for SQLite: downloads](https://sqlitebrowser.org/dl/)
- [DB Browser: gebruikershandleiding](https://github.com/sqlitebrowser/sqlitebrowser/wiki)
- [SQLite: gegevenstypen](https://www.sqlite.org/datatype3.html)
- [SQLite: tabellen en beperkingen](https://www.sqlite.org/lang_createtable.html)
- [Python: SQLite-back-upfunctie](https://docs.python.org/3/library/sqlite3.html#sqlite3.Connection.backup)

De schermafbeeldingen zijn eigen opnames van de gebruikte Windows-interface. Ze zijn bijgesneden om alleen de relevante bediening te tonen.

</div>